# OpenAI Responses API

## What is the OpenAI Responses API?

The Responses API is a new API released in March 2025. It is a combination of the traditional 
Chat Completions API and the Assistants API, providing support for:

- **Traditional Chat Completions:** Facilitates seamless conversational AI experiences.
- **Web Search:** Enables real-time information retrieval from the internet.
- **File Search:** Allows searching within files for relevant data.

Accordingly, the Assistants API will be retired in 2026. 

> **For new users, OpenAI recommends using the Responses API instead of the Chat Completions API to leverage its expanded capabilities.**

For a comprehensive comparison between the Responses API and the Chat Completions API, refer to the official OpenAI documentation: 
[Responses vs. Chat Completions](https://platform.openai.com/docs/guides/responses-vs-chat-completions).

## Summary of This Notebook
This notebook provides a hands-on guide for using the **OpenAI Responses API** to analyze tweets. 
It covers essential techniques such as:

- **Creating a vector store** and uploading tweets for semantic search.
- **Using file search** to analyze private datasets.
- **Performing a web search** to retrieve the latest public information.
- **Utilizing stateful responses** to maintain conversation context.
- **Combining file and web search** to enhance retrieval-augmented generation (RAG) applications.

By the end of this notebook, users will be able to integrate OpenAI's Responses API for efficient data retrieval and analysis of structured and unstructured data.

## Install Required Libraries
To use the OpenAI Responses API, we need to install the following libraries:

- **`openai`**: Provides access to OpenAI's APIs, including the Responses API

In [7]:
pip install openai -q

Note: you may need to restart the kernel to use updated packages.


## Import Required Libraries

In [8]:
from IPython.display import Markdown, display
import boto3
from botocore.exceptions import ClientError
import json
import io

## Retrieve Secrets from AWS Secrets Manager

In [9]:
def get_secret(secret_name):
    region_name = "us-east-1"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        raise e

    secret = get_secret_value_response['SecretString']
    
    return json.loads(secret)

## Initialize OpenAI Client

In [10]:
from openai import OpenAI
openai_api_key  = get_secret('openai')['api_key']

client = OpenAI(api_key=openai_api_key)

## File Search API

### Introduction to File Search
File search API enables efficient retrieval of relevant information 
from uploaded files by leveraging vector-based indexing. This feature is particularly useful 
for searching large datasets, extracting insights, and improving retrieval-augmented generation (RAG) applications.

Unlike traditional keyword-based searches, the Responses API uses embeddings 
to identify semantically relevant content, making it ideal for analyzing structured 
and unstructured text data (OpenAI, 2025).

For more details, visit the official OpenAI documentation: 
[File Search in Responses API](https://platform.openai.com/docs/guides/tools-file-search).

### Create a Vector Store

In [11]:
vector_store = client.vector_stores.create(
    name="my_vector_store"
)
vector_store_id = vector_store.id
print(vector_store_id)

vs_69178178e58c819198537a412e4ea62f


### Upload Files

In [12]:
with open('tweet_text (3).json', 'rb') as f:
    file = client.files.create(
        file=f,            # file-like object
        purpose="assistants"
    )

file_id = file.id
print(file_id)

file-Ud62G2diSuSe4SPisQzb4n


### Attach File to Vector Store

In [13]:
attach_status =client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_id
            )

print(attach_status.id)

file-Ud62G2diSuSe4SPisQzb4n


### Query the Vector Store

In [14]:
query = "the latest development in generativeAI"

In [15]:
search_results = client.vector_stores.search(
    vector_store_id=vector_store_id,
    query=query
)

for result in search_results.data[:5]:
    print(result.content[0].text[:100] + '\n Relevant score: ' + str(result.score))

## OpenAI Response API

### Simple Response

In [16]:
simple_response = client.responses.create(
  model="gpt-4o",
  input=[
      {
          "role": "user",
          "content": query
      }
  ]
)

In [17]:
display(Markdown(simple_response.output_text))

As of the latest updates, generative AI has seen several exciting developments:

1. **Enhanced Multimodal Models**: New models are capable of processing and generating content across multiple types of data, such as text, images, and audio, simultaneously. This convergence is enhancing applications in creative fields, human-computer interaction, and more.

2. **Improved Efficiency and Scale**: Advances in training methodologies and hardware have led to more efficient models that require less computational power while delivering high performance. This has made it feasible to deploy powerful AI on consumer-grade devices.

3. **Better Understanding and Control**: Researchers are focusing on creating models with better interpretability and controllability, allowing users to fine-tune outputs more accurately and mitigate biases.

4. **Ethical and Safe AI**: There's ongoing work on frameworks to ensure responsible use of generative AI, focusing on reducing harmful outputs and promoting fairness and transparency.

5. **Open-Source Initiatives**: An increase in open-source projects and collaborative platforms is driving innovation and making advanced generative AI models accessible to a broader audience.

6. **Industry-Specific Applications**: Tailored generative AI solutions are emerging across various sectors, such as healthcare, entertainment, and marketing, providing domain-specific insights and creative solutions.

These developments indicate a rapidly evolving landscape where generative AI continues to push the boundaries of creativity and utility.

### File Search Response

In [18]:

file_search_response = client.responses.create(
    input= query,
    model="gpt-4o",
    temperature = 0,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id],
    }]
)

In [19]:
display(Markdown(file_search_response.output_text))


The latest developments in generative AI include several exciting advancements:

1. **OpenAI's Sora2**: This tool allows users to create cinematic videos from a prompt, now with added features like audio, physics, and cameos, offering endless creativity for creators.

2. **Generative AI in Business**: AI is being used to revolutionize how businesses innovate, from content generation to design systems.

3. **AI in Creative Industries**: Generative AI is transforming creative industries by enabling the creation of cinematic food commercials and other media content without traditional production resources.

4. **AI in Healthcare**: Generative AI is being explored to support doctors with data interpretation and preliminary analysis, highlighting its potential in transforming healthcare.

These developments illustrate the broad and transformative impact of generative AI across various sectors.

## Web Search API

### Introduction to Web Search
The OpenAI Web Search tool allows models to retrieve real-time information from the internet. 
This capability is particularly useful for obtaining up-to-date data, fact-checking, and expanding knowledge 
without relying solely on pre-trained information. 

By leveraging OpenAI's web search functionality, the Responses API can fetch external data 
and provide accurate, relevant results in real time (OpenAI, 2025). 
This feature enhances applications that require the latest insights, such as news aggregation, research, 
or dynamic content generation.

For more details, visit the official OpenAI documentation: 
[Web Search in Responses API](https://platform.openai.com/docs/guides/tools-web-search).

### Perform Web Search

In [20]:
web_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= query,
    tools=[
        {
            "type": "web_search"
        }
    ]
)

In [21]:
display(Markdown(web_search_response.output_text))

Below is a detailed and structured exploration of the *latest developments in generative AI* as of today, **November 14, 2025**, highlighting cutting-edge innovations, emerging trends, and real-world applications across sectors. Each paragraph includes multiple citations to ensure accuracy and currency.

---

##  1. OpenAI Launches GPT‑5.1 with Enhanced Personalities and Reasoning

OpenAI unveiled **GPT‑5.1** on **November 12, 2025**, marking its most recent and advanced model in the GPT series. This release improves upon GPT‑5 by reducing hallucinations, enhancing instruction compliance, increasing speed, and introducing eight distinct personality options for users. It features two modes—**Instant** for quick responses and **Thinking** for deeper reasoning tasks—highlighting a refined mechanism for tailoring AI interactions. ([en.wikipedia.org](https://en.wikipedia.org/wiki/GPT-5.1?utm_source=openai))

This model is accessible through both **ChatGPT** and **Microsoft Copilot**, continuing OpenAI’s strategy of integrating advanced generative AI seamlessly into productivity platforms. ([en.wikipedia.org](https://en.wikipedia.org/wiki/GPT-5.1?utm_source=openai))

---

##  2. Google’s "Nano Banana" (Gemini 2.5 Flash Image) Goes Viral

Google DeepMind’s **"Nano Banana"**, officially known as **Gemini 2.5 Flash Image**, was released in **August 2025** as part of Google’s Gemini AI ecosystem. Its standout feature is photorealistic image editing: users can modify hairstyles, switch backdrops, and remix photos via natural language. The model also supports subject consistency across edits, multi-image fusion, and embeds invisible SynthID watermarks to authenticate AI-generated content. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Nano_Banana?utm_source=openai))

The novelty captured public imagination, going viral on social media and drawing over **10 million new users** to the Gemini app within weeks. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Nano_Banana?utm_source=openai))

---

##  3. Edge-Based Generative AI Moves into Industrial Automation

On **November 13, 2025**, **Rockwell Automation** announced the integration of **NVIDIA Nemotron Nano (Nemotron‑Nano‑9B‑v2)** into its **FactoryTalk Design Studio**. This edge-deployed generative AI brings intelligence directly to industrial equipment—such as HMI panels and offline environments—empowering smarter, real-time decision-making in manufacturing workflows. The solution supports low-power and air-gapped deployments, making it suitable for on-premises and unreliable-network settings. Rockwell plans to demonstrate the technology at the **Automation Fair** taking place **November 17–20, 2025**. ([stocktitan.net](https://www.stocktitan.net/news/ROK/rockwell-automation-to-advance-industrial-intelligence-through-edge-8x58h6sr82la.html?utm_source=openai))

This development marks a broader shift toward embedding generative AI at the “edge,” enhancing autonomy and resilience in industrial IoT applications.

---

##  4. AI-Assisted Public Services: Anthropic AI in Maryland

In a move that demonstrates the growing societal role of generative AI, the state of **Maryland** announced on **November 13, 2025** that it will use **Anthropic’s AI tools** to address critical public policy issues, including **child poverty** and **housing access**. These tools aim to streamline administrative processes and improve resource targeting, reflecting a real-world application of generative AI in governance. ([statescoop.com](https://statescoop.com/maryland-ai-anthropic-housing-access-child-poverty/?utm_source=openai))

---

##  5. Broader Industry Trends and Academic Innovations

- **Multimodal Unification**: Generative AI models increasingly handle text, images, audio, and video within a single architecture. Notable advances in 2025 include:
  - **Google’s Veo 3**, a text-to-video system generating videos complete with sound and dialogue. ([aiinsight.blog](https://aiinsight.blog/generative-ai-advancements-in-2025?utm_source=openai))
  - **OpenAI’s Sora**, integrated into ChatGPT as a native video generation tool for Plus users. ([aiinsight.blog](https://aiinsight.blog/generative-ai-advancements-in-2025?utm_source=openai))
  - **Midjourney V7**, which brought 3D model generation and text-to-video capabilities to its platform. ([aiinsight.blog](https://aiinsight.blog/generative-ai-advancements-in-2025?utm_source=openai))

- **Software Development Revolution**: Generative AI is transforming coding workflows. Notable shifts in 2025 include:
  - Models like **GPT‑4o**, **Copilot X**, and **Anthropic Claude 3** jointly enable tasks such as generating microservices, refactoring code, and embedding security insights into pull requests. ([thetechthinker.com](https://thetechthinker.com/generative-ai-in-software-development-2025/?utm_source=openai))

- **Open Innovation and Agent Orchestration**:
  - Platforms like **Hugging Face’s SmolLM3** and **Liquid AI’s LFM2** deliver efficient, open-source LLMs optimized for on-device and edge deployment. ([linkedin.com](https://www.linkedin.com/pulse/top-generative-ai-updates-week-july-2-2025-kalyan-ks-oorrc?utm_source=openai))
  - Trend toward multi-model orchestration: coordinating diverse AI models into cohesive pipelines to share context and manage error correction is growing in 2025. ([futureagi.com](https://futureagi.com/blogs/generative-ai-trends-2025?utm_source=openai))

- **Academic Advances**:
  - Researchers introduced the concept of **Chronologically Consistent Generative AI**, where models are trained strictly on data preceding a defined cutoff to prevent lookahead bias—a boon for replicability and fair forecasting. ([arxiv.org](https://arxiv.org/abs/2510.11677?utm_source=openai))
  - Another study proposes **Interactive Generative Video** as the backbone for next-generation, AI-driven **game engines**, enabling dynamic content creation with reasoning and memory capabilities. ([arxiv.org](https://arxiv.org/abs/2503.17359?utm_source=openai))

---

##  Summary and Outlook

As of **November 14, 2025**, the latest landscape of generative AI features:

- OpenAI’s **GPT‑5.1**, offering refined reasoning and personality options.
- Google’s **Nano Banana**, a viral multimodal image editing model.
- Industrial adoption of **edge generative AI** through Rockwell Automation.
- Public sector use of AI via **Anthropic tools in Maryland**.
- Rapid progress in **multimodal systems**, coding assistance, **open-source LLMs**, orchestration architectures, and academic frameworks ensuring temporal integrity and interactive content generation.

These developments signify a turning point where generative AI is not only expanding in technical sophistication—combining modalities, deploying at the edge, enabling agentic behaviors—but also permeating critical domains like manufacturing, governance, creativity, and software development.

If you’d like, I can delve deeper into any of these specific advances, spotlight emerging companies, or explore regulatory and societal impacts. Let me know what interests you next.

### Stateful Response

The OpenAI Responses API includes a stateful feature that enables continuity in interactions. 
By using the `response_id`, a conversation can persist across multiple queries, 
allowing users to refine or expand upon previous searches. This is particularly useful for iterative research, 
dynamic content generation, and applications that require follow-up queries based on prior responses.

In [22]:
fetched_response = client.responses.retrieve(response_id=web_search_response.id)
display(Markdown(fetched_response.output_text[:100]))

Below is a detailed and structured exploration of the *latest developments in generative AI* as of t

### Continue Query with Web Search

In [23]:
continue_query = 'find different news'

continue_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= continue_query,
    previous_response_id=web_search_response.id,
    tools=[
        {
            "type": "web_search"
        }
    ]
)

In [24]:
display(Markdown(continue_search_response.output_text))

Here are several **recent newsworthy developments** in the field of *generative AI*, covering creative tools, enterprise adoption, accuracy & regulation, media innovation, and security. Each section is richly sourced and structured for clarity.

---

## Adobe Unleashes AI Innovations Across Creative Cloud (Late October 2025)   
Adobe introduced a sweeping set of **AI-assisted features** during its Adobe MAX conference on **October 28, 2025**, enriching flagship Creative Cloud tools like Photoshop, Illustrator, Lightroom, and Premiere. These innovations are designed to aid creative professionals by streamlining workflows and delivering enhanced precision and control. ([news.adobe.com](https://news.adobe.com/news/2025/10/adobe-max-2025-creative-cloud?utm_source=openai))

---

## MIT Researchers Enable Personalized Object Localization in Vision-Language Models (October 2025)  
An interdisciplinary team from **MIT**, the MIT-IBM Watson AI Lab, and the **Weizmann Institute of Science** developed a novel training technique that allows generative vision-language models to accurately **localize personalized objects**—for example, identifying a user's specific cat across different images—addressing a notable limitation in current systems. ([news.mit.edu](https://news.mit.edu/2025/method-teaches-generative-ai-models-locate-personalized-objects-1016?utm_source=openai))

---

## YouTube Rolls Out AI Creation Tools for Shorts, Including Veo 3 with Sound (Mid‑September 2025)  
In mid-September, YouTube unveiled new generative AI features as part of its "Made on YouTube" initiative. The updates include:
 • **Veo 3 Fast** – A text-to-video generation tool that now includes **accompanying audio**, optimized for seamless mobile Shorts creation.  
 • **Edit with AI** – Automatically selects and arranges key footage, adds music, transitions, and even a voiceover in English or Hindi to create polished Shorts. ([blog.youtube](https://blog.youtube/news-and-events/generative-ai-creation-tools-made-on-youtube-2025/?utm_source=openai))

---

## TIME Launches the TIME AI Agent—A Conversational AI for News Engagement (Early November 2025)  
**TIME Magazine** introduced the **TIME AI Agent**, a conversational platform powered by generative AI developed in collaboration with Scale AI. Launched just **four days ago**, it enables users to interact dynamically with editorial content—generating summaries, audio reports, translations, and more—while adhering to rigorous standards of editorial integrity, moderation, and transparency. ([time.com](https://time.com/7332572/the-story-behind-the-time-ai-agent/?utm_source=openai))

---

## AI Chatbots Show High Error Rate on News Content, Study Finds (Late October–Mid November 2025)  
A recent international study found that approximately **45% of news-related responses** from AI chatbots are inaccurate. **Google’s Gemini** model notably underperformed, generating erroneous responses in about **76%** of cases. ([computerworld.com](https://www.computerworld.com/article/4077344/ai-chatbots-are-wrong-about-news-45-of-the-time.html?utm_source=openai))

---

## AWS and Reuters Showcase Generative AI for News Distribution (September 2025)  
At IBC 2025, **AWS**, in collaboration with **Reuters**, showcased how **generative AI** can transform the production, exchange, and monetization of news. They demonstrated integrations with **Amazon Bedrock** and **TAMS**, plus video understanding models by TwelveLabs, illustrating how cloud-native AI systems can reshape journalistic workflows. ([aws.amazon.com](https://aws.amazon.com/blogs/media/aws-to-show-new-generative-ai-news-distribution-innovations-at-ibc-2025/?utm_source=openai))

---

###  Summary Table

| Domain             | Notable Development                                                                 |
|--------------------|-------------------------------------------------------------------------------------|
| Creativity         | Adobe embeds generative AI across Creative Cloud tools.                            |
| Vision-Language AI | MIT enables personalization in object localization across varied scenes.           |
| Social Media       | YouTube adopts Veo 3 with sound and AI editing for Shorts creators.                |
| Journalism         | TIME introduces an AI agent for interactive news consumption.                      |
| Accuracy Concern   | Study reveals high error rates in chatbot-generated news content.                  |
| News Infrastructure| AWS and Reuters explore AI-powered newsroom workflows at IBC 2025.                 |

---

These developments highlight the rapid integration and expansion of generative AI across creative industries, journalism, enterprise tools, and media, accompanied by growing scrutiny around reliability and ethical use.

If any of these topics pique your interest, I’d be happy to dive deeper into technical specifics, business implications, or societal impact—just let me know!

### Combining File Search and Web Search

This is an example of using file search to analyze private data and web search to retrieve public or the latest data. 
The Responses API allows developers to integrate these tools to enhance retrieval-augmented generation (RAG) applications. 
By combining file search with web search, users can leverage structured internal knowledge while also retrieving real-time 
information from external sources, ensuring comprehensive and up-to-date responses. 

In [25]:
combined_search_response = client.responses.create(
    model="gpt-4o",  # or another supported model
    input= query,
    temperature = 0,
    instructions="Retrieve the results from the file search first, and use the web search tool to expand the results with news resources",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_id],
    },
        {
            "type": "web_search"
        }
    ]
)

In [26]:
display(Markdown(combined_search_response.output_text))

Here’s a comprehensive and up-to-date overview of the **latest developments in generative AI** as of November 14, 2025. This analysis draws on recent news, model releases, market trends, and emerging research, with citations for each key point.

---

##  Major Model Releases and Innovations

- **OpenAI GPT‑5.1**  
  Released on **November 12, 2025**, GPT‑5.1 is the newest iteration in OpenAI’s GPT series. It introduces eight selectable personality options, improved instruction-following, reduced hallucinations, and two operational modes: *Instant* for speed and *Thinking* for complex reasoning tasks ([en.wikipedia.org](https://en.wikipedia.org/wiki/GPT-5.1?utm_source=openai)).

- **OpenAI GPT‑5**  
  Launched earlier on **August 7, 2025**, GPT‑5 unified reasoning and multimodal capabilities under a single interface. It became available via ChatGPT, Microsoft Copilot, and the OpenAI API ([en.wikipedia.org](https://en.wikipedia.org/wiki/GPT-5?utm_source=openai)).

- **OpenAI o4‑mini**  
  Released on **April 16, 2025**, this compact reasoning model supports both text and image inputs, including whiteboard sketch analysis. A higher-accuracy variant, *o4‑mini‑high*, is available to paid-tier users ([en.wikipedia.org](https://en.wikipedia.org/wiki/OpenAI_o4-mini?utm_source=openai)).

- **Google Gemini 2.5 Family & “Nano Banana”**  
  Google’s Gemini 2.5 Pro and Flash models, introduced in early 2025, offer advanced reasoning, coding, and multimodal capabilities, including a “thinking model” and a 1-million-token context window ([en.wikipedia.org](https://en.wikipedia.org/wiki/Gemini_%28language_model%29?utm_source=openai)).  
  The image-focused variant, nicknamed **Nano Banana** (Gemini 2.5 Flash Image), launched publicly on **August 26, 2025**. It enables photorealistic 3D-style edits, multi-image fusion, and includes SynthID watermarking. It quickly went viral, attracting over 10 million new users and facilitating more than 200 million image edits ([en.wikipedia.org](https://en.wikipedia.org/wiki/Nano_Banana?utm_source=openai)).

- **Other Notable Models**  
  - **GPT‑4.5 (Orion)**: Released February 2025, this model offers enhanced accuracy, reduced hallucinations, and a large context window, excelling in creative and emotionally intelligent tasks ([medium.com](https://medium.com/%4017shubhanshujain/top-generative-ai-trends-for-2025-gpt-4-5-gemini-2-5-and-ethical-implications-94a126df84d8?utm_source=openai)).  
  - **Llama 4 (Meta)**: Released April 2025, this open-source family includes Scout, Maverick, and Behemoth variants, featuring early-fusion multimodality, Mixture-of-Experts architecture, and support for over 200 languages ([medium.com](https://medium.com/%4017shubhanshujain/top-generative-ai-trends-for-2025-gpt-4-5-gemini-2-5-and-ethical-implications-94a126df84d8?utm_source=openai)).  
  - **Baidu Ernie 4.5 & Ernie X1**: Announced in April 2025, these multimodal and reasoning models reportedly outperform competitors on benchmarks like CCBench and OCRBench. Ernie 4.5 is slated to become open-source from **June 30, 2025** ([globenewswire.com](https://www.globenewswire.com/news-release/2025/04/17/3063915/0/en/Applied-Generative-AI-Course-Launched-by-Interview-Kickstart-2025-Best-GenAI-Course-With-Agentic-AI-Projects-For-Top-AI-Jobs-at-Google-Meta-Netflix-Microsoft-OpenAI-Nvidia.html?utm_source=openai)).

---

##  Emerging Trends and Applications

- **Agentic AI and Autonomous Agents**  
  The shift from AI as copilots to fully autonomous agents is accelerating. Agentic AI systems can plan, act, and adapt independently—handling tasks like campaign testing and budget adjustments without human intervention. The autonomous AI market is projected to reach **USD 11.79 billion by 2026**, growing at over 40% CAGR ([simplilearn.com](https://www.simplilearn.com/top-technology-trends-and-jobs-article?utm_source=openai)).

- **Enterprise Adoption and Scaling**  
  According to the **2025 McKinsey Global Survey on AI**, 23% of organizations are scaling agentic AI systems, while 39% are experimenting with them. High-performing companies are more likely to redesign workflows, invest over 20% of digital budgets in AI, and embed AI into business processes with strong governance ([mckinsey.com](https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai?utm_source=openai)).

- **Generative AI in Software Development**  
  A **Bain & Company** report (September 23, 2025) highlights that while generative AI tools boost productivity, realizing real value requires process changes and integration into development workflows ([bain.com](https://www.bain.com/insights/from-pilots-to-payoff-generative-ai-in-software-development-technology-report-2025/?utm_source=openai)).

- **Responsible AI Practices**  
  UC Berkeley’s Responsible Use of Generative AI playbook outlines ten actionable strategies—five for business leaders and five for product managers—to ensure ethical and responsible deployment of GenAI in daily operations and product development ([weforum.org](https://www.weforum.org/stories/2025/06/responsible-generative-ai-product-development-use/?utm_source=openai)).

---

##  Market Growth and Infrastructure

- **Generative AI Market Expansion**  
  A report from **ResearchAndMarkets.com** indicates that in 2024, the generative AI market experienced triple-digit growth across hardware, foundation models, and development platforms. AI-related spending is expected to exceed **US$400 billion in 2025** ([businesswire.com](https://www.businesswire.com/news/home/20250825682581/en/Generative-AI-Market-Report-2025-GenAI-Market-Experienced-Triple-digit-growth-Rates-in-All-Three-Major-Segments-Spanning-GenAI-Hardware-Foundation-Models-and-Development-Platforms---ResearchAndMarkets.com?utm_source=openai)).

- **Hardware Innovations**  
  **Qualcomm** recently unveiled the **AI200** and **AI250** inference accelerators, targeting data center deployments in 2026 and 2027. These systems feature advanced NPUs, high memory capacity, liquid cooling, and support for major AI frameworks, signaling Qualcomm’s strategic entry into AI infrastructure ([tomshardware.com](https://www.tomshardware.com/tech-industry/artificial-intelligence/qualcomm-unveils-ai200-and-ai250-ai-inference-accelerators-hexagon-takes-on-amd-and-nvidia-in-the-booming-data-center-realm?utm_source=openai)).

---

##  Research Highlights

- **Chronologically Consistent Generative AI**  
  A recent arXiv paper (October 13, 2025) introduces models trained only on data available before a defined cutoff, eliminating lookahead bias. These models offer replicability and conservative forecast accuracy, useful for prediction tasks ([arxiv.org](https://arxiv.org/abs/2510.11677?utm_source=openai)).

- **Agentic AI in Networking (AgentNet)**  
  Another study (March 2025) proposes **AgentNet**, a framework for agentic AI in 6G networks. It envisions generative foundation models acting as autonomous agents that collaborate, learn, and adapt in dynamic environments—applicable to industrial automation and metaverse infotainment systems ([arxiv.org](https://arxiv.org/abs/2503.15764?utm_source=openai)).

---

##  Summary Table

| Category                  | Key Developments                                                                 |
|---------------------------|----------------------------------------------------------------------------------|
| Model Releases            | GPT‑5.1, GPT‑5, o4‑mini, Gemini 2.5, Nano Banana, GPT‑4.5, Llama 4, Ernie 4.5/X1 |
| Trends & Applications     | Agentic AI, enterprise scaling, software dev integration, responsible AI         |
| Market & Infrastructure   | Triple-digit market growth, Qualcomm AI accelerators                             |
| Research Innovations      | Chronologically consistent models, AgentNet for 6G networks                      |

---

### Final Thoughts

The generative AI landscape in late 2025 is defined by rapid innovation across models, infrastructure, and applications. OpenAI and Google continue to push the boundaries with GPT‑5.1 and Gemini 2.5, while open-source and international players like Meta and Baidu contribute to a diverse ecosystem. Agentic AI is emerging as a transformative trend, with enterprises beginning to scale autonomous systems. Meanwhile, responsible AI frameworks and hardware advancements are laying the groundwork for sustainable, scalable, and ethical AI deployment.

Let me know if you'd like a deeper dive into any specific model, trend, or application area!

# 🧩 Try It Yourself: Two-Step RAG (Private Data + Combined Search)

## Step 1 — Upload & Create Vector Store
1. Upload a short text file (e.g., `my_notes.txt`) to your notebook instance.  
2. Create a **vector store** and **ingest** your uploaded file.  
3. Run a simple test query to verify retrieval:  

In [28]:
tiy_vector_store = client.vector_stores.create(
    name="tiy_vector_store"
)
tiy_vector_store_id = tiy_vector_store.id
print("Vector store created:", tiy_vector_store_id)

with open("343 article.txt", "rb") as f:
    uploaded_file = client.files.create(
        file=f,
        purpose="assistants"
    )

print("File uploaded:", uploaded_file.id)


attach_result = client.vector_stores.files.create(
    vector_store_id=tiy_vector_store_id,
    file_id=uploaded_file.id,
)
print("File attached to vector store:", attach_result.id)


test_query = "Summarize the main ideas from my notes."
search_results = client.vector_stores.search(
    vector_store_id=tiy_vector_store_id,
    query=test_query,
)

print("\nTop retrieved chunks:")
for item in search_results.data[:3]:
    print("-" * 40)
    print(item.content[0].text.strip())

Vector store created: vs_69178209eb448191bfd583a7a9bc7048
File uploaded: file-2ePkGYogPue5B1HXsoiwLu
File attached to vector store: file-2ePkGYogPue5B1HXsoiwLu

Top retrieved chunks:


## Step 2 — Combine File Search with Web Search
1. Enable both **file_search** and **web_search** in the Responses API.  
2. Use a prompt that asks the model to merge insights from both sources.  
   > Example: “Using my uploaded notes and the latest web information, summarize the current trends on this topic.”  
3. Review how the answer from your file and **current info** from the web.

✅ You’ve created a RAG system that combines **private** and **public** data for comprehensive, up-to-date analysis.


In [29]:
query = (
    "Using the uploaded news article as the primary source, and also checking the latest web information, "
    "give me a short, structured summary of the topic (El Paso Texas and change following immigration crackdown). Clearly separate 'From my file' vs 'From the web'."
)

combined_response = client.responses.create(
    model="gpt-4o",
    input=query,
    temperature=0,
    instructions=(
        "First look up relevant passages from the attached vector store. "
        "Then augment with web_search to bring in current/public info. "
        "Present the answer in two sections: 'From my file(s)' and 'From the web'."
    ),
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [tiy_vector_store_id],
        },
        {
            "type": "web_search"
        }
    ],
)

display(Markdown(combined_response.output_text))

### From my file

The article discusses the impact of Texas' Operation Lone Star, launched in 2021, which increased border enforcement with state troopers, the National Guard, and miles of razor wire. This operation has led to more migrants taking dangerous routes to enter the U.S., resulting in a rise in deaths in the New Mexico and West Texas desert. The article highlights the human cost of these policies, with many bodies remaining unidentified for months. The global surge in migration is noted, with the number of immigrants worldwide doubling since 1990. The article criticizes both state and federal officials for not addressing the issue adequately, with Texas Governor Abbott's office blaming President Biden's policies for the increase in migrant deaths.

### From the web

I'll now look up the latest information on this topic. Please hold on.Here is a structured summary of the topic—**El Paso, Texas, and changes following the immigration crackdown**—divided into two sections as requested:

From my file  
----------------  
- The article centers on **Operation Lone Star**, initiated by Texas in 2021, which significantly ramped up border enforcement using state troopers, the National Guard, and extensive razor wire barriers.  
- As a result, migrants have increasingly taken perilous routes through the New Mexico and West Texas deserts, leading to a rise in deaths. Many of the deceased remain unidentified for extended periods.  
- The article underscores the **human cost** of these policies and criticizes both state and federal officials for failing to address the crisis effectively. Governor Abbott’s office attributes the surge in migrant deaths to President Biden’s immigration policies.  

From the web  
----------------  
- **Dramatic decline in migrant encounters**: In Fiscal Year 2025, migrant encounters in the U.S. Southwest border dropped by 84.5%, from 1.53 million in 2024 to 237,538. Specifically, the El Paso Sector saw an 81.6% decrease. Border Patrol recorded 47,165 encounters in FY2025, down from 256,102 in FY2024 and 427,471 in FY2023. Rescues and deaths also declined significantly. ([mrt.com](https://www.mrt.com/news/article/southwest-border-encounters-fall-21123168.php?utm_source=openai))  

- **Expansion of enforcement infrastructure**: A new ICE mega-detention facility, dubbed the “Lone Star Lockup,” opened at Fort Bliss in El Paso. It currently holds 1,000 detainees and is planned to expand to 5,000 beds, backed by a $1 billion investment. The facility is part of a broader federal push for mass deportations and has drawn criticism over transparency, cost, and potential human rights concerns. ([time.com](https://time.com/7310657/ice-immigration-detention-texas-lone-star-lockup/?utm_source=openai))  

- **Increased military involvement**: Texas has deployed 500 National Guard soldiers to El Paso to support border surveillance, infrastructure, and logistics. Some are deputized under Title 8, enabling them to make arrests under federal immigration law. This militarization has raised legal and civil liberties concerns. ([elpais.com](https://elpais.com/us/migracion/2025-09-15/texas-intensifica-su-control-fronterizo-con-el-despliegue-de-500-soldados-de-la-guardia-nacional-en-el-paso.html?utm_source=openai))  

- **Financial strain on El Paso County**: The county has incurred substantial costs—over $10 million—for processing and housing state inmates under Operation Lone Star. A disaster declaration was issued to seek state reimbursement, as the county projects up to $18 million in lost federal revenue due to reduced jail space for federal inmates. ([kvia.com](https://kvia.com/news/border/2024/07/22/operation-lone-star-el-paso-county-lost-10-million/?utm_source=openai))  

- **Human rights concerns**: Human Rights Watch reports that Operation Lone Star has led to injuries, deaths, racial discrimination, and suppression of civil liberties. Allegations include the use of pepper spray projectiles, rubber bullets, and physical assaults by National Guard members. ([hrw.org](https://www.hrw.org/news/2025/02/13/us-texas-vehicle-pursuits-kill-least-106-injure-301?utm_source=openai))  

- **Symbolic and political responses**: In October 2025, the El Paso bishop delivered letters from fearful migrants to Pope Leo XIV, highlighting the emotional and spiritual toll of the crackdown. The Pope expressed solidarity with migrants and church leaders. ([spectrumlocalnews.com](https://spectrumlocalnews.com/tx/south-texas-el-paso/news/2025/10/08/pope-leo-xiv-migrant-letters?utm_source=openai))  

- **Reopening of community spaces**: Shelby Park in Eagle Pass, previously closed and used as a base for migrant arrests, has been reopened. This move reflects a minor rollback in border enforcement, though broader operations continue. ([houstonchronicle.com](https://www.houstonchronicle.com/politics/texas/article/shelby-park-greg-abbott-operation-lone-star-20263037.php?utm_source=openai))  

Summary  
----------------  
Operation Lone Star has transformed El Paso’s border landscape through aggressive enforcement, military deployment, and expanded detention capacity. These measures have coincided with a sharp decline in migrant encounters and crossings. However, they have also imposed significant financial burdens on local authorities, raised serious human rights and legal concerns, and deeply affected migrant communities. The situation remains dynamic, with both enforcement and humanitarian dimensions evolving rapidly.

In [ ]:
from openai import OpenAI
client = OpenAI()

# 1. Create a new vector store (only needs to be done once)
vector_store = client.beta.vector_stores.create(
    name="el_paso_article_store"
)

# 2. Upload your file into the vector store
#    (replace the filename if yours differs)
file_path = "/mnt/data/343 article.txt"

file = client.files.create(
    file=open(file_path, "rb"),
    purpose="assistants"
)

# 3. Add that file to the vector store
client.beta.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file.id
)

# 4. Your original query
query = (
    "Using the uploaded news article as the primary source, and also checking the "
    "latest web information, give me a short, structured summary of the topic "
    "(El Paso Texas and change following immigration crackdown). Clearly separate "
    "'From my file' vs 'From the web'."
)

# 5. Run the combined response with both tools
combined_response = client.responses.create(
    model="gpt-4o",
    input=query,
    temperature=0,
    instructions=(
        "First search relevant passages from the vector store. "
        "Then supplement with web_search. "
        "Present answer with two clearly separated sections: "
        "'From my file(s)' and 'From the web'."
    ),
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id],
        },
        {
            "type": "web_search"
        }
    ],
)

# 6. Print the answer
print(combined_response.output_text)
